# Continued Pre-Training

A practical refresher on **Continued Pre-Training (CPT)** — also called *continual* or
*continued* pre-training, and in its domain-targeted form *Domain-Adaptive Pre-Training
(DAPT)*. CPT takes an **already-pretrained** foundation model (Llama, Mistral, Qwen, …)
and runs **more self-supervised pre-training** on a new corpus — a domain (medicine,
law, code), a language, or simply fresher data — **before** any instruction tuning or
RLHF. It is the cheap middle ground between training from scratch (millions of dollars)
and parameter-efficient fine-tuning (good for style/format, weak for new *knowledge*).

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction <a id="introduction"></a>

### What is it?

**Continued Pre-Training** resumes the *same* next-token (causal LM) objective the base
model was originally trained with, but on a **new, unlabeled corpus**. No human labels,
no prompt/response pairs — just raw text the model learns to predict. The result is a
new **base** model whose internal representations have shifted toward your data
distribution, which you then (optionally) instruction-tune and align.

Where the typical LLM pipeline is **pre-train → SFT → preference-align**, CPT inserts an
extra stage at the front: **pre-train → *continued pre-train* → SFT → align**.

### Why use it?

- **Inject genuine new knowledge.** Unlike LoRA/SFT, which mostly teach *format and
  behavior*, CPT updates the dense weights on billions of new tokens, so the model
  actually *knows* more about your domain, jargon, and entities.
- **Adapt to a new language or script** the base model saw little of (e.g. continued
  pre-training Llama on Korean or Arabic corpora).
- **Refresh stale knowledge** by training on data newer than the base model's cutoff.
- **Expand or specialize the tokenizer** (new domain vocab, a new language's
  characters) and let the model relearn the new embeddings.
- **Far cheaper than from-scratch**: you reuse a multi-million-dollar base model and
  spend tens-to-hundreds of GPU-hours instead of millions.

### When to use it?

- You have a **large** (≳1B tokens) high-quality, *unlabeled* domain corpus.
- The base model demonstrably **lacks knowledge** you need (high perplexity on your
  text, hallucinated domain facts) — not just the wrong tone or output format.
- SFT/RAG alone underperform because the gaps are in *what the model knows*, not *how it
  responds*.
- You can tolerate a **base** model output that still needs SFT afterward to be useful as
  an assistant.

## Key Features <a id="key-features"></a>

### What distinguishes CPT from other adaptation methods

| Capability | What it means | Why it matters |
|------------|---------------|----------------|
| **Self-supervised, label-free** | Trains on raw text with the causal-LM loss; no annotation needed | You can use the huge pile of unlabeled internal docs you already have |
| **Knowledge injection** | Updates dense weights over billions of tokens | Adds facts/terminology, not just style — the gap SFT can't close |
| **Distribution shift** | Re-centers the model on your domain/language | Lower perplexity and better downstream SFT on in-domain tasks |
| **Tokenizer extension** | Add domain/language tokens and train the new embeddings | Fewer tokens per document → cheaper inference, better morphology |
| **Resumable / checkpointable** | Same objective and optimizer state as base pre-training | Restart from any step; spot-instance friendly |
| **Composable** | Produces a new *base* model for downstream SFT/LoRA/RLHF | Slots cleanly into an existing post-training pipeline |

## Architecture Overview <a id="architecture"></a>

CPT is the same transformer, the same loss, a new corpus and a carefully chosen
**lower** learning rate. The pipeline:

```
                    ┌────────────────────────────────────────────────┐
  Raw domain text → │  Data prep: clean → dedup → quality filter →    │
  (PDFs, wiki,      │  (optional) tokenizer extension → tokenize →    │
   code, tickets)   │  pack into fixed-length sequences (e.g. 4096)   │
                    └───────────────────────┬────────────────────────┘
                                            │ packed token shards (.bin / Arrow)
                                            ▼
  Base checkpoint ──► ┌──────────────────────────────────────────────┐
  (Llama-3-8B,        │  Continued pre-training loop                  │
   load weights +     │  • causal-LM loss (predict next token)       │
   tokenizer)         │  • low LR + warmup + cosine/constant decay   │
                      │  • replay mix: domain + some general data    │
                      │  • FSDP / DeepSpeed ZeRO-3 sharding          │
                      └───────────────────────┬──────────────────────┘
                                              │ new BASE model
                                              ▼
                       SFT (instruction tuning) → preference alignment → serve
```

### Components

1. **Data pipeline** — the part that actually decides quality. Cleaning, near-dedup
   (MinHash/SimHash), quality filtering, optional tokenizer extension, tokenization, and
   **sequence packing** (concatenate documents to fill every position; use an
   attention-reset / document mask so packed docs don't attend across boundaries).
2. **Training engine** — the base model loaded under a sharding strategy
   (**PyTorch FSDP** or **DeepSpeed ZeRO-3**) so an 8–70B model fits across GPUs, plus
   activation checkpointing, mixed precision (bf16), and gradient accumulation.
3. **Replay buffer** — a slice of *general* pre-training-style data mixed into the domain
   corpus to fight **catastrophic forgetting** (typically 1–30% general data).
4. **Evaluation harness** — held-out **perplexity** on domain + general text, plus
   downstream task evals, run on a cadence to catch forgetting early.

## Installation <a id="installation"></a>

### Prerequisites

- **Multi-GPU host(s)** with enough aggregate VRAM. Rule of thumb for full-parameter
  bf16 training: ~**16–20 GB per billion parameters** (weights + gradients + Adam
  states + activations). An 8B model needs roughly 8×A100/H100-80GB with FSDP; LoRA-style
  continued pre-training fits on far less but injects less knowledge.
- **CUDA 12.x**, PyTorch 2.x, and NCCL configured for your interconnect (NVLink /
  EFA / InfiniBand).
- Libraries: `transformers`, `datasets`, `accelerate`, and an engine —
  **`torchtitan`/FSDP**, **`deepspeed`**, or a turnkey trainer like
  **`llm-foundry`**, **Axolotl**, or **NeMo**.
- A **tokenized, packed** dataset on fast shared storage (FSx for Lustre, local NVMe).

### Installation steps

The runnable cells below are **pure-Python planners/estimators** (no GPUs needed). The
commented block shows the real environment install; uncomment it on a training box.

In [ ]:
# Real training environments install something like the following (run on a GPU box):
#
# !pip install "torch>=2.3" --index-url https://download.pytorch.org/whl/cu121
# !pip install transformers datasets accelerate deepspeed flash-attn --no-build-isolation
#
# The cells in THIS notebook are pure-Python and need no GPU or extra packages.
import sys
print("Python", sys.version.split()[0])
print("CPT planning cells run anywhere; the actual training runs on a multi-GPU host.")

## Basic Usage <a id="basic-usage"></a>

The smallest real CPT loop is "load a base model, keep training it on your packed corpus
with a low learning rate." Below is a minimal HuggingFace + `Trainer` sketch.

```python
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          Trainer, TrainingArguments, DataCollatorForLanguageModeling)
from datasets import load_from_disk

base = "meta-llama/Meta-Llama-3-8B"
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype="bfloat16",
                                             attn_implementation="flash_attention_2")

# Pre-tokenized + packed into fixed 4096-token blocks ahead of time.
ds = load_from_disk("/data/domain_packed_4096")

args = TrainingArguments(
    output_dir="/ckpt/llama3-8b-cpt",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,     # → larger effective batch
    learning_rate=2e-5,                 # LOW vs ~3e-4 from-scratch
    lr_scheduler_type="cosine",
    warmup_ratio=0.01,
    bf16=True,
    gradient_checkpointing=True,
    max_steps=20000,
    save_steps=500,
    logging_steps=10,
    deepspeed="ds_zero3.json",          # or use FSDP via accelerate
)

collator = DataCollatorForLanguageModeling(tok, mlm=False)  # causal LM
Trainer(model=model, args=args, train_dataset=ds, data_collator=collator).train()
```

Launch across GPUs with `deepspeed --num_gpus 8 train.py` or
`accelerate launch --config_file fsdp.yaml train.py`. The pure-Python cell below
estimates the **compute and wall-clock** before you ever rent the GPUs.

In [ ]:
# Estimate CPT compute (FLOPs), GPU-hours and rough cost from token budget + model size.
# Rule of thumb for transformer training: ~6 * N_params * N_tokens FLOPs (fwd+bwd).

def cpt_estimate(params_b, tokens_b, gpus, gpu_tflops=312, mfu=0.4, gpu_cost_per_hr=2.0):
    """params_b: model size in billions; tokens_b: training tokens in billions.
    gpu_tflops: peak bf16 TFLOP/s (A100=312, H100=~990); mfu: model-FLOP utilization."""
    flops = 6 * (params_b * 1e9) * (tokens_b * 1e9)
    eff_flops_per_s = gpus * gpu_tflops * 1e12 * mfu
    seconds = flops / eff_flops_per_s
    gpu_hours = (seconds / 3600) * gpus
    return {
        "total_PFLOPs": round(flops / 1e15, 1),
        "wallclock_hours": round(seconds / 3600, 1),
        "gpu_hours": round(gpu_hours, 1),
        "est_cost_usd": round(gpu_hours * gpu_cost_per_hr, 0),
    }

# Continue-pretrain an 8B model on 10B domain tokens, 8x A100-80GB.
print("8B model, 10B tokens, 8xA100:", cpt_estimate(8, 10, gpus=8))
print("8B model, 50B tokens, 8xH100:", cpt_estimate(8, 50, gpus=8, gpu_tflops=990))

## Advanced Features <a id="advanced-features"></a>

### 1. Replay to fight catastrophic forgetting

The core risk of CPT is **catastrophic forgetting**: optimizing hard on a narrow corpus
erodes the general capabilities the base model spent trillions of tokens acquiring. The
standard mitigation is **replay** — mix a fraction of *general* pre-training-style data
(web/wiki/code) into every batch. Common ratios are **5–30% general** to
**70–95% domain**. Higher general fraction = safer but slower domain adaptation.

### 2. Learning-rate schedule (the #1 knob)

Re-warming a converged model with a *high* LR causes a "loss spike" and damages the base
weights. Use a **low peak LR** (often **1e-5 to 3e-5**, i.e. ~10× lower than from-scratch
~3e-4), a **short warmup** (0.5–2% of steps), and a cosine or **constant-then-decay**
schedule. For long runs an **infinite/WSD (warmup-stable-decay)** schedule lets you
extend training and only decay at the end.

### 3. Tokenizer extension

To add a language or domain vocabulary, **extend** the tokenizer with new tokens and
resize the embedding matrix. Initialize each new row as the **mean of the subword
embeddings** the new token decomposes into (rather than random) so the model starts from
a sensible point and converges faster.

### 4. Parameter-efficient CPT

When full-parameter training is too expensive, **LoRA/QLoRA** continued pre-training
trains low-rank adapters over the frozen base. It costs a fraction of the memory but
injects **less** knowledge than full-parameter CPT — good for budget-constrained domain
nudges, not deep adaptation.

In [ ]:
# Build a domain/replay/general mixing plan and convert it to a per-step token budget.

def mixing_plan(total_tokens_b, domain_frac=0.85, general_frac=0.15,
                seq_len=4096, global_batch_seqs=512):
    assert abs(domain_frac + general_frac - 1.0) < 1e-6, "fractions must sum to 1"
    tokens_per_step = seq_len * global_batch_seqs
    total_tokens = total_tokens_b * 1e9
    steps = round(total_tokens / tokens_per_step)
    return {
        "domain_tokens_B": round(total_tokens_b * domain_frac, 2),
        "general_replay_tokens_B": round(total_tokens_b * general_frac, 2),
        "tokens_per_step": tokens_per_step,
        "total_steps": steps,
        "general_seqs_per_batch": round(global_batch_seqs * general_frac),
    }

print(mixing_plan(10, domain_frac=0.85, general_frac=0.15))
print(mixing_plan(10, domain_frac=0.95, general_frac=0.05))  # more aggressive adaptation

## Use Cases <a id="use-cases"></a>

### Use Case 1: Domain-adaptive pre-training for finance

- **Context**: a base model hallucinates on SEC filings, ticker semantics, and
  accounting standards. RAG helps retrieval but the model still reasons poorly about the
  domain.
- **Implementation**: CPT on ~30B tokens of 10-Ks, earnings calls, research notes, and
  regulatory text, with 15% general replay; then SFT on Q&A pairs. (This is the recipe
  behind models like **BloombergGPT**, which mixed a large finance corpus with general
  data.)
- **Results**: lower perplexity on financial text and materially better downstream SFT
  accuracy on in-domain tasks, with general benchmarks roughly preserved thanks to replay.

### Use Case 2: New-language adaptation

- **Context**: Llama-class models are English-heavy and tokenize, say, Korean
  inefficiently (many tokens per word) and reason worse in it.
- **Implementation**: extend the tokenizer with Korean tokens, mean-init the new
  embeddings, then CPT on a large Korean corpus with English replay to retain English.
- **Results**: fewer tokens per Korean document (cheaper inference) and large gains on
  Korean evals; this is the pattern behind community models like **Llama-Ko / SOLAR**-style
  efforts.

### Use Case 3: Code-domain adaptation

- **Context**: a general base model is mediocre on an internal language/framework.
- **Implementation**: CPT on a deduplicated internal+public code corpus packed with
  document-boundary masking, then instruction-tune for chat/completion.
- **Results**: better completion and fewer API hallucinations for the target stack.

## Best Practices <a id="best-practices"></a>

1. **Spend your effort on data, not hyperparameters.** Dedup aggressively (near-dup
   removal with MinHash), filter low-quality and boilerplate text, and remove eval-set
   contamination. Data quality dominates outcomes far more than the LR schedule.
2. **Use a low peak learning rate with short warmup.** ~1e-5 to 3e-5 with a 0.5–2%
   warmup avoids the loss spike that damages base weights when you re-warm a converged
   model.
3. **Always mix in general replay data** (5–30%) to preserve broad capabilities. Track
   general perplexity to confirm you're not forgetting.
4. **Pack sequences with document-boundary attention masking.** Packing fills every
   token position (cheap), but without a reset mask the model attends across unrelated
   documents and learns spurious correlations.
5. **Mean-initialize extended tokenizer embeddings** from constituent subwords; never
   leave them random if you can avoid it.
6. **Checkpoint frequently and keep optimizer state.** CPT runs are long and often on
   spot/preemptible GPUs; resumability is not optional.
7. **Evaluate on a fixed held-out set on a cadence** — domain perplexity *and* general
   benchmarks — so you can stop at the best trade-off, not the last step.
8. **Remember CPT produces a *base* model.** Plan the downstream SFT + alignment stage;
   the CPT output is rarely a usable assistant on its own.

## Common Pitfalls <a id="pitfalls"></a>

1. **Catastrophic forgetting.** Training too hard on a narrow corpus, with no replay and
   too high an LR, wrecks general ability. *Fix*: add replay, lower the LR, watch general
   perplexity.
2. **Learning-rate too high / no warmup → loss spike.** Re-warming a converged model at
   from-scratch LRs spikes the loss and can corrupt the weights irrecoverably. *Fix*:
   low peak LR, short warmup, gradient clipping.
3. **Packing without a document mask.** Concatenated docs attend across boundaries,
   teaching the model nonsense joins. *Fix*: attention-reset / `position_ids` reset per
   document (a.k.a. intra-document masking).
4. **Data leakage / benchmark contamination.** Your corpus contains eval data, inflating
   scores. *Fix*: decontaminate against known benchmarks before training.
5. **Duplicates inflate "tokens" but not learning.** Heavy near-duplication wastes
   compute and overfits. *Fix*: MinHash/SimHash near-dedup first.
6. **Expecting CPT to teach behavior.** CPT injects knowledge, not instruction-following
   or safety. *Fix*: do SFT + alignment afterward; don't ship the raw CPT base.
7. **Tokenizer mismatch.** Extending the tokenizer but forgetting to resize embeddings
   (or training with the *old* tokenizer) silently breaks training.

## Performance Optimization <a id="performance"></a>

### Configuration tuning

- **Sharding strategy**: **FSDP** or **DeepSpeed ZeRO-3** shard params/grads/optimizer
  states so large models fit; ZeRO-3 + CPU/NVMe offload trades speed for the ability to
  fit on fewer GPUs.
- **Sequence packing**: keep packing efficiency high (≈100% token utilization) so you
  pay for compute, not padding.
- **Mixed precision**: train in **bf16** (stable, no loss-scaling headaches vs fp16);
  keep a master copy in fp32 inside the optimizer.
- **Activation/gradient checkpointing**: recompute activations to cut memory at ~20–30%
  extra compute — usually worth it to raise batch size.
- **FlashAttention-2**: large speed/memory win for long sequences.
- **Effective batch size**: raise via gradient accumulation to a stable large global
  batch (hundreds of sequences) for smooth loss.
- **Maximize MFU**: model-FLOP utilization of 0.35–0.55 on A100/H100 is a realistic
  target; below ~0.3 you're leaving money on the table (check data loading, comms).

### The metric that matters: tokens/sec/GPU

Throughput, not GPU count, sets wall-clock. The cell below converts a target
tokens/sec/GPU into a wall-clock estimate so you can budget a run.

In [ ]:
# Convert achieved throughput (tokens/sec/GPU) into wall-clock and a simple MFU check.

def throughput_plan(params_b, tokens_b, gpus, tok_per_s_per_gpu, gpu_tflops=312):
    total_tok = tokens_b * 1e9
    agg_tok_per_s = tok_per_s_per_gpu * gpus
    seconds = total_tok / agg_tok_per_s
    # Achieved FLOP/s vs peak → model-FLOP utilization (MFU).
    achieved_flops = 6 * (params_b * 1e9) * agg_tok_per_s
    mfu = achieved_flops / (gpus * gpu_tflops * 1e12)
    return {
        "wallclock_hours": round(seconds / 3600, 1),
        "wallclock_days": round(seconds / 86400, 2),
        "implied_MFU": round(mfu, 3),
    }

# 8B model, 50B tokens, 32xA100, ~3,000 tok/s/GPU.
print(throughput_plan(8, 50, gpus=32, tok_per_s_per_gpu=3000))

## Production Deployment <a id="deployment"></a>

CPT is a **training job**, so "deployment" means wiring it into a scheduler and storage
on a GPU cluster, then promoting the resulting checkpoint into the post-training pipeline.

#### Containerized training image

```dockerfile
FROM nvcr.io/nvidia/pytorch:24.05-py3
RUN pip install --no-cache-dir transformers datasets accelerate deepspeed     flash-attn --no-build-isolation
COPY train_cpt.py ds_zero3.json /workspace/
WORKDIR /workspace
ENTRYPOINT ["deepspeed", "--num_gpus", "8", "train_cpt.py"]
```

#### Kubernetes multi-GPU Job (single 8-GPU node)

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: llama3-8b-cpt
spec:
  backoffLimit: 4                 # resume after spot interruption
  template:
    spec:
      restartPolicy: OnFailure
      containers:
        - name: trainer
          image: <registry>/llama-cpt:latest
          resources:
            limits:
              nvidia.com/gpu: 8
          volumeMounts:
            - { name: data, mountPath: /data }       # packed token shards (FSx/Lustre)
            - { name: ckpt, mountPath: /ckpt }        # checkpoints (S3-backed PVC)
            - { name: dshm, mountPath: /dev/shm }     # NCCL/shared memory
      volumes:
        - { name: data, persistentVolumeClaim: { claimName: domain-tokens } }
        - { name: ckpt, persistentVolumeClaim: { claimName: cpt-checkpoints } }
        - { name: dshm, emptyDir: { medium: Memory, sizeLimit: 16Gi } }
```

For **multi-node** runs use the Kubeflow **PyTorchJob** (or Volcano/MPI operator) so the
operator wires up `MASTER_ADDR`, `WORLD_SIZE`, and `RANK` across pods, and pin the network
to EFA/InfiniBand. Always **checkpoint to durable storage** (S3/GCS) on a step cadence so a
preemption resumes instead of restarting.

## Monitoring and Observability <a id="monitoring"></a>

#### Key metrics to track

- **Training loss & gradient norm** — the primary health signal; a sudden loss *spike* or
  exploding grad-norm means LR/warmup is wrong. Stop and fix.
- **Held-out domain perplexity** (going down = adapting) **and held-out general
  perplexity** (rising sharply = forgetting). Track both, every N steps.
- **Tokens/sec/GPU and MFU** — throughput and efficiency; a drop signals data-loader
  stalls or comms bottlenecks.
- **GPU utilization, memory, NVLink/NCCL bandwidth, ECC errors** — via DCGM/`nvidia-smi`.
- **Learning rate** — confirm the schedule (warmup → decay) is actually being applied.
- **Checkpoint write latency / cadence** — ensure resumability is real.

#### Logging best practices

- Stream scalars to **Weights & Biases / TensorBoard / MLflow**; log a downstream eval
  (not just loss) periodically so "loss down" can't hide capability regressions.
- Emit **structured run metadata** — base model + revision, data mix and version, seed,
  LR schedule, global batch — so a run is reproducible and auditable.
- Export **DCGM** GPU metrics to Prometheus and alert on grad-norm spikes, throughput
  cliffs, and NaN/Inf loss.

## Troubleshooting <a id="troubleshooting"></a>

#### Issue 1: Loss spikes or diverges early in training

**Symptoms**: loss jumps sharply in the first hundreds of steps, grad-norm explodes, or
you see `NaN`/`Inf`.

**Cause**: peak LR too high for a converged model, warmup too short/absent, or fp16
instability.

**Solution**: drop peak LR (try 1e-5), add/lengthen warmup, clip gradients
(`max_grad_norm=1.0`), and train in **bf16** instead of fp16.

#### Issue 2: General-benchmark scores collapse after CPT

**Symptoms**: domain perplexity improves but MMLU/general evals drop a lot — classic
catastrophic forgetting.

**Cause**: too narrow a corpus, no replay, too many epochs, or LR too high.

**Solution**: add 10–30% general replay data, lower the LR, train fewer tokens/epochs,
and stop at the best trade-off checkpoint (you tracked general perplexity, so you have it).

#### Issue 3: Out-of-memory (CUDA OOM)

**Symptoms**: `CUDA out of memory` at start or when sequence length / batch grows.

**Cause**: model + optimizer states don't fit; activations too large.

**Solution**: switch to **FSDP/ZeRO-3** (shard optimizer states), enable gradient
checkpointing, lower per-device batch and raise gradient accumulation, enable
CPU/NVMe offload, or shorten the sequence length.

#### Issue 4: Throughput far below expected (low MFU)

**Symptoms**: tokens/sec/GPU well under the cell's estimate; GPUs idle in `nvidia-smi`.

**Cause**: data-loader starvation, slow storage, poor packing, or comms-bound sharding.

**Solution**: pre-tokenize and pack offline, use fast shared storage + more dataloader
workers, raise packing efficiency, and verify NCCL is using NVLink/EFA not TCP.

## Comparison with Alternatives <a id="comparison"></a>

### How CPT compares to other ways of specializing a model

| Dimension | **Continued Pre-Training** | SFT / Instruction Tuning | LoRA / QLoRA | RAG | Pre-train from scratch |
|-----------|----------------------------|--------------------------|--------------|-----|------------------------|
| Data needed | Large *unlabeled* corpus | Labeled prompt/response pairs | Small labeled set | A document index | Trillions of tokens |
| Mainly teaches | **Knowledge + distribution** | Behavior / format | Behavior / light knowledge | Retrieval-time facts | Everything |
| Cost | Medium (10s–100s GPU-hrs) | Low–medium | **Low** | Very low (no training) | **Very high ($M)** |
| Updates dense weights | **Yes (all)** | Yes (all) | Adapters only | No | Yes |
| Forgetting risk | Medium–high (needs replay) | Medium | Low | None | n/a |
| Output | A new **base** model | An assistant | Adapter + base | Same model + index | A base model |

### When to choose CPT

- Your gap is **knowledge**, not format, and a **large unlabeled** domain/language corpus
  exists.
- SFT and RAG alone underperform because the model fundamentally doesn't *know* the domain.
- You can afford a real training run and a follow-on SFT/alignment stage.

Choose **RAG** for fast-changing facts, **SFT/LoRA** for behavior/format on a budget, and
**from-scratch** only when no suitable base model exists. CPT and these are
**complementary** — the strongest domain models often do CPT → SFT → align → serve with RAG.

## Resources <a id="resources"></a>

### Papers & foundational reading

- *Don't Stop Pretraining: Adapt Language Models to Domains and Tasks* (Gururangan et
  al., 2020) — the DAPT/TAPT paper that established domain-adaptive pre-training.
- *Simple and Scalable Strategies to Continually Pre-train Large Language Models*
  (Ibrahim et al., 2024) — LR re-warming, replay ratios, and avoiding forgetting at scale.
- *BloombergGPT* (Wu et al., 2023) — a large finance model mixing domain + general data.
- *Scaling Laws for Neural Language Models* (Kaplan et al., 2020) — the
  `6 · N · D` FLOPs intuition used in the estimator cells.

### Tools & frameworks

- **MosaicML LLM Foundry** — production CPT/pre-training recipes and packing utilities.
- **NVIDIA NeMo** / **Megatron-LM** — large-scale tensor/pipeline-parallel training.
- **PyTorch FSDP** and **DeepSpeed ZeRO** — sharded full-parameter training.
- **HuggingFace `transformers` + `accelerate`** and **Axolotl** — accessible trainers.
- **`datatrove` / `nemo-curator`** — corpus cleaning, dedup, and tokenization at scale.

### Evaluation & ops

- **`lm-evaluation-harness`** (EleutherAI) — standardized downstream evals.
- **Weights & Biases / MLflow / TensorBoard** — experiment tracking.
- **NVIDIA DCGM + Prometheus/Grafana** — GPU fleet observability for long runs.

### Related techniques

- Domain-Adaptive Pre-Training (DAPT) and Task-Adaptive Pre-Training (TAPT)
- Supervised Fine-Tuning (SFT) and preference alignment (DPO/RLHF)
- Parameter-efficient fine-tuning (LoRA/QLoRA), tokenizer extension, and RAG